# 24 · State-Space Models and Modern Sequence Architectures

- **Level:** Modern (2020 → present)
- **Typical semester:** research
- **Prerequisites:** 05 (§6 linear systems), 09 (Z-transform, FFT), 18, 19
- **Matches code in:** `lipipy.research`


## 1. Continuous-time linear state-space model

$$h'(t) = Ah(t) + Bx(t), \qquad y(t) = Ch(t) + Dx(t)$$
$h\in\mathbb R^N$ is the hidden state, $x$ and $y$ are scalar input and output channels (applied per channel).


## 2. Discretization

With step size $\Delta$:

**Zero-order hold (ZOH):** $\bar A = e^{\Delta A}$, $\;\bar B = (\Delta A)^{-1}(e^{\Delta A} - I)\Delta B$

**Bilinear (Tustin):** $\bar A = (I - \frac\Delta2A)^{-1}(I + \frac\Delta2A)$, $\;\bar B = (I - \frac\Delta2A)^{-1}\Delta B$

**Simplified (as used in Mamba for $B$):** $\bar B\approx\Delta B$

Recurrence: $h_t = \bar Ah_{t-1} + \bar Bx_t$, $\;y_t = Ch_t$


## 3. Convolutional view

Unrolling the recurrence (with $h_{-1} = 0$):
$$y_t = \sum_{k=0}^{t}C\bar A^k\bar Bx_{t-k}, \qquad \bar K = (C\bar B, C\bar A\bar B, C\bar A^2\bar B, \dots, C\bar A^{L-1}\bar B), \qquad y = \bar K * x$$
Training uses the parallel convolution (via FFT, $O(L\log L)$); inference uses the $O(1)$-per-step recurrence.


## 4. HiPPO and S4

**HiPPO-LegS** matrix (optimal online projection onto Legendre polynomials):
$$A_{nk} = -\begin{cases}\sqrt{2n+1}\sqrt{2k+1} & n > k\\ n + 1 & n = k\\ 0 & n < k\end{cases}, \qquad B_n = \sqrt{2n+1}$$

**S4:** parameterize $A$ as normal plus low-rank (NPLR), $A = V\Lambda V^* - PQ^\top$, and compute the kernel through its generating function using the Woodbury identity and Cauchy kernels.

**S4D / DSS:** diagonal $A = \text{diag}(\lambda_n)$ (e.g. S4D-Lin $\lambda_n = -\frac12 + i\pi n$); kernel $\bar K_\ell = \sum_nC_n\bar B_n\bar\lambda_n^\ell$ is a Vandermonde product.

**S5:** a single multi-input multi-output SSM with diagonal $A$, computed by a parallel associative scan.


## 5. Parallel scan

A linear recurrence $h_t = a_th_{t-1} + b_t$ is an associative operation on pairs:
$$(a_1, b_1)\bullet(a_2, b_2) = (a_1a_2,\; a_2b_1 + b_2)$$
so all $h_t$ can be computed in $O(\log L)$ parallel steps (Blelloch scan). This makes input-dependent (time-varying) recurrences trainable in parallel.


## 6. Mamba (selective SSM)

Make $B, C, \Delta$ functions of the input:
$$B_t = \text{Linear}_B(x_t), \quad C_t = \text{Linear}_C(x_t), \quad \Delta_t = \text{softplus}(\text{Linear}_\Delta(x_t) + b_\Delta)$$
$$\bar A_t = \exp(\Delta_tA), \quad \bar B_t = \Delta_tB_t, \quad h_t = \bar A_th_{t-1} + \bar B_tx_t, \quad y_t = C_th_t$$
Large $\Delta_t$ means "reset and focus on the current input"; small $\Delta_t$ means "ignore it and keep the state". Computed with a hardware-aware parallel scan. A Mamba block wraps this with a causal conv1d and a SiLU gate.

**Mamba-2 / structured state-space duality (SSD):** with scalar $A_t = a_tI$, the SSM equals masked linear attention
$$y = (L\circ CB^\top)x, \qquad L_{ij} = \begin{cases}\prod_{k=j+1}^{i}a_k & i\ge j\\0 & i<j\end{cases}$$
so it can be computed with matrix multiplies in chunks.


## 7. Linear attention family as recurrences

| Model | State update | Output |
|---|---|---|
| Linear attention | $S_t = S_{t-1} + k_tv_t^\top$ | $o_t = S_t^\top q_t$ |
| RetNet | $S_t = \gamma S_{t-1} + k_tv_t^\top$ | $o_t = S_t^\top q_t$ |
| Gated linear attention (GLA) | $S_t = \text{diag}(\alpha_t)S_{t-1} + k_tv_t^\top$ | $o_t = S_t^\top q_t$ |
| DeltaNet | $S_t = S_{t-1} - \beta_tk_t(k_t^\top S_{t-1} - v_t^\top)$ | $o_t = S_t^\top q_t$ |
| Gated DeltaNet | $S_t = \alpha_t(I - \beta_tk_tk_t^\top)S_{t-1} + \beta_tk_tv_t^\top$ | $o_t = S_t^\top q_t$ |

The delta rule is one step of online gradient descent on $\frac12\|S^\top k_t - v_t\|^2$ (a "fast-weight" memory). Test-time-training (TTT) layers generalize this: the state is the weights of a small model updated by gradient steps.

**RetNet** has three equivalent forms: parallel $\left(QK^\top\odot D\right)V$ with $D_{nm} = \gamma^{n-m}[n\ge m]$, recurrent, and chunk-wise.


## 8. RWKV

Time-mixing with exponential decay $w$ and a bonus $u$ for the current token:
$$wkv_t = \frac{\sum_{i=1}^{t-1}e^{-(t-1-i)w + k_i}v_i + e^{u + k_t}v_t}{\sum_{i=1}^{t-1}e^{-(t-1-i)w + k_i} + e^{u + k_t}}, \qquad o_t = W_o\left(\sigma(r_t)\odot wkv_t\right)$$
Later versions make the decay data-dependent and use matrix-valued states.


## 9. Long convolutions: Hyena

$y = h_N * (x_N\odot(h_{N-1} * (\cdots(x_1\odot(h_1 * v)))))$ with implicitly parameterized long filters $h_i(t) = \text{FFN}(\text{PE}(t))\cdot\text{window}(t)$, evaluated by FFT in $O(L\log L)$.


## 10. xLSTM

**sLSTM:** exponential gating $i_t = \exp(\tilde i_t)$ with a normalizer state $n_t = f_tn_{t-1} + i_t$, $h_t = o_t\odot\frac{c_t}{n_t}$, stabilized by a running max.

**mLSTM:** matrix memory $C_t = f_tC_{t-1} + i_tv_tk_t^\top$, $\;n_t = f_tn_{t-1} + i_tk_t$, $\;h_t = o_t\odot\frac{C_tq_t}{\max(|n_t^\top q_t|, 1)}$


## 11. Continuous-depth and other alternatives

**Neural ODE:** $\frac{dh}{dt} = f_\theta(h, t)$, $h(T) = h(0) + \int_0^Tf_\theta\,dt$ (trained with the adjoint method, see 16 §7)

**Liquid time-constant networks:** $\frac{dx}{dt} = -\left[\frac1\tau + f(x, I, \theta)\right]x + f(x, I, \theta)A$

**Kolmogorov-Arnold representation theorem:** $f(x_1,\dots,x_n) = \sum_{q=0}^{2n}\Phi_q\left(\sum_{p=1}^n\phi_{q,p}(x_p)\right)$. **KAN layer:** learnable univariate functions (B-splines) on edges, $x_{l+1,j} = \sum_i\phi_{l,j,i}(x_{l,i})$.


## 12. Complexity summary

| Architecture | Training | Inference per token | State size |
|---|---|---|---|
| Softmax attention | $O(L^2d)$ | $O(Ld)$ (KV cache) | grows with $L$ |
| Linear attention / RetNet / GLA | $O(Ld^2)$ | $O(d^2)$ | fixed |
| S4 / S5 / Mamba | $O(LdN)$ (scan) or $O(L\log L)$ (FFT) | $O(dN)$ | fixed |
| Hyena | $O(L\log L)$ | $O(L)$ without distillation | grows |

Hybrids (attention layers interleaved with SSM / linear-attention layers) trade off exact recall against constant-memory inference.
